# Lecture: Controllable Generation II — ControlNet

In C4-7 we controlled generation through the **starting image** (img2img) or a
**mask** (inpainting). Both keep us tied to an existing image's pixels. But often
we want **structural** control that a prompt cannot express: "put a person in
*exactly this pose*", "match *this* layout", "follow *these* edges" — while still
generating freely within that structure.

**ControlNet** (Zhang et al., 2023) adds exactly this. It augments a frozen Stable
Diffusion model with a second, trainable branch that takes a **control image** — an
edge map, a human pose skeleton, a depth map, a segmentation — and conditions every
denoising step on it. The result follows the *structure* of the control image while
the *content* is driven by the text prompt.

The key idea, building on everything so far:

> ControlNet is another **conditioning signal** (like the text prompt in C4-4, or
> the input image in C4-7), but a **spatial** one. It is injected into the same
> U-Net you know, leaving the base model's weights frozen.

In this notebook we use the most common variant — **Canny edge** control — to:

1. Generate a base image and extract its edges.
2. Generate a completely new image that **follows those edges** but matches a new
   prompt.
3. Vary the **control strength** to trade structural fidelity against prompt
   freedom.

> **Requirements:** GPU runtime (Colab: T4). Downloads the base SD model plus a
> ControlNet model (~1.4 GB extra).

### Install dependencies

Same pinned base versions as C4-6 / C4-7, plus **OpenCV** for Canny edge detection
(usually already present on Colab).

In [ ]:
!pip install -q diffusers==0.31.0 transformers==4.44.2 accelerate==0.34.2 opencv-python

### Load the ControlNet + Stable Diffusion pipeline

We load a **Canny-edge ControlNet** and attach it to the familiar SD 1.5 base
model via `StableDiffusionControlNetPipeline`. The base model is the same one from
C4-6; ControlNet is the extra conditioning branch.

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, StableDiffusionPipeline

assert torch.cuda.is_available(), "This notebook requires a GPU runtime (Colab: T4)."
device = "cuda"

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    controlnet=controlnet, torch_dtype=torch.float16, safety_checker=None,
).to(device)

print("ControlNet (Canny) + Stable Diffusion loaded.")

### Step 0 — Create a base image and extract its edges

To stay self-contained we first generate a base image with plain text2img, then run
**Canny edge detection** on it. The edge map is the *control image* — a black image
with white outlines that captures the structure but none of the colour or texture.

In [ ]:
# Generate a base image with the plain text2img pipeline.
base_pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16, safety_checker=None,
).to(device)

base_prompt = "a photograph of a vintage motorcycle parked on a street, side view"
generator = torch.Generator(device=device).manual_seed(8)
base_image = base_pipe(base_prompt, num_inference_steps=50, guidance_scale=7.5,
                       generator=generator).images[0]

In [ ]:
# Canny edge detection: convert to a structural outline.
image_np = np.array(base_image)
edges = cv2.Canny(image_np, threshold1=100, threshold2=200)
edges_rgb = np.stack([edges] * 3, axis=-1)        # ControlNet expects a 3-channel image
canny_image = Image.fromarray(edges_rgb)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(base_image); axes[0].set_title("base image", fontsize=11); axes[0].axis("off")
axes[1].imshow(canny_image); axes[1].set_title("Canny edges (control image)", fontsize=11); axes[1].axis("off")
plt.tight_layout()
plt.show()

## Generate a new image that follows the edges

Now the core of ControlNet: we give the pipeline the **edge map** as structural
guidance and a **new prompt** describing different content. The output keeps the
*shape* of the original motorcycle but re-imagines it according to the prompt — a
control that no text prompt alone could achieve.

In [ ]:
control_prompt = ("a futuristic chrome motorcycle, neon lights, cyberpunk city, "
                  "highly detailed, sharp focus")

generator = torch.Generator(device=device).manual_seed(0)
result = pipe(control_prompt, image=canny_image, num_inference_steps=50,
              guidance_scale=7.5, generator=generator).images[0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
axes[0].imshow(base_image);  axes[0].set_title("original", fontsize=11); axes[0].axis("off")
axes[1].imshow(canny_image); axes[1].set_title("edge control", fontsize=11); axes[1].axis("off")
axes[2].imshow(result);      axes[2].set_title("new image (follows edges)", fontsize=11); axes[2].axis("off")
plt.tight_layout()
plt.show()

### The control strength

`controlnet_conditioning_scale` sets how strongly the edge map constrains the
output:

- **low** (≈ 0.3): edges are a loose suggestion; the prompt dominates and the
  structure may drift.
- **high** (≈ 1.5): the output rigidly follows every edge, sometimes at the cost of
  image quality.
- **default** (1.0): a balanced compromise.

We sweep it for the same prompt, seed and edge map. This is the spatial analogue of
the `guidance_scale` (C4-4) and `strength` (C4-7) knobs — another dial on the
control-vs-freedom trade-off.

In [ ]:
scales = [0.3, 0.7, 1.0, 1.5]

fig, axes = plt.subplots(1, len(scales), figsize=(16, 4.5))
for ax, cs in zip(axes, scales):
    generator = torch.Generator(device=device).manual_seed(0)
    img = pipe(control_prompt, image=canny_image, num_inference_steps=50,
               guidance_scale=7.5, controlnet_conditioning_scale=cs,
               generator=generator).images[0]
    ax.imshow(img); ax.set_title(f"control scale = {cs}", fontsize=11); ax.axis("off")
plt.suptitle("ControlNet: higher control scale follows the edges more rigidly", y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### Same structure, different content

The real power of ControlNet: **one structure, many outputs**. Keeping the same
edge map, we change only the prompt to generate entirely different images that all
share the original layout. This is how ControlNet is used in practice — fix a
composition once, then explore content freely.

In [ ]:
prompts = [
    "a red classic motorcycle, sunny day, photograph",
    "a wooden toy motorcycle, studio product shot",
    "a glowing neon motorcycle, dark background, digital art",
]

fig, axes = plt.subplots(1, len(prompts), figsize=(16, 5.5))
for ax, p in zip(axes, prompts):
    generator = torch.Generator(device=device).manual_seed(1)
    img = pipe(p, image=canny_image, num_inference_steps=50, guidance_scale=7.5,
               generator=generator).images[0]
    ax.imshow(img); ax.set_title(p[:28] + "...", fontsize=9); ax.axis("off")
plt.suptitle("One edge map, three prompts — shared structure, different content", y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Summary

ControlNet completes the picture of **conditioning** built up across the course:

| Conditioning | Notebook | Controls |
|---|---|---|
| Text prompt + CFG | C4-4 / C4-6 | *what* content |
| Input image + `strength` | C4-7 (img2img) | overall appearance |
| Mask | C4-7 (inpainting) | *where* to edit |
| **Control image (edges/pose/depth)** | **C4-8 (ControlNet)** | *spatial structure* |

All of these inject a signal into the **same** frozen diffusion U-Net from C4 — they
differ only in *what* signal and *how* it enters. ControlNet's contribution is a
trainable side-branch that conditions on a full spatial map, giving precise control
over composition that text alone never could.

Beyond Canny edges, the same mechanism powers pose control (OpenPose), depth
control, scribble-to-image and segmentation-to-image — just swap the ControlNet
model and the type of control image.

---
## Try It Yourself — Structural Control

Work in pairs. **Predict first, then run, then explain in one sentence.**

**A. Tune the control scale.** In the control-scale sweep, which value best balances
"follows the bike's shape" against "looks like a good image"? Relate this to the
`guidance_scale` (C4-4) and `strength` (C4-7) knobs — what is the common trade-off?

**B. Change the Canny thresholds.** Lower `threshold1`/`threshold2` in the edge
detector. How does a denser edge map change the output? Why does the *amount* of
detail in the control image matter as much as the control scale?

**C. One structure, your prompts.** Add your own prompts to the "same structure,
different content" cell. Which prompts respect the edges well, and which fight
against them? What kinds of content are hard to fit into a fixed structure?

**D. Where does the signal enter?** ControlNet leaves the base SD weights frozen and
adds a side branch. In one sentence, explain why this design lets a single base
model support many different ControlNets (edges, pose, depth) without retraining it.

**E. Connect the chapter.** Across C4-7 and C4-8 you have four ways to control
generation (strength, mask, edges, prompt). For a concrete task — "replace the sky
in a landscape photo but keep the mountains' outline" — which combination would you
use, and why?